# 03 Neural stream inventory and decoder manifest

This notebook bridges the gap between the trial catalog and the first decoding notebook.

## Purpose
1. Open every NWB file listed in the session index.
2. Inventory acquisition and processing objects across sessions.
3. Identify candidate neural streams for decoding.
4. Build a decoder manifest that maps each session to a likely neural source.
5. Save figures, tables, and metadata for the baseline decoder notebook.

## Inputs
- `/kaggle/working/tables_session_index/session_master_index.csv`
- `/kaggle/working/tables_session_index/decoder_trial_table.csv`
- Raw NWB files under:
  `/kaggle/input/datasets/katakuricharlotte/dandi-dataset/001201/sub-Monkey-N`

## Outputs
- `tables_stream_inventory/all_object_manifest.csv`
- `tables_stream_inventory/candidate_stream_manifest.csv`
- `tables_stream_inventory/session_decoder_stream_map.csv`
- `meta_stream_inventory/stream_inventory_report.json`
- `figures_stream_inventory/*.png`

In [2]:
!pip install pynwb h5py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 17.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.9/341.9 kB 25.1 MB/s eta 0:00:00


In [3]:
import os
import re
import json
import math
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from pynwb import NWBHDF5IO

In [4]:
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "axes.linewidth": 1.2,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "grid.linestyle": ":",
    "grid.linewidth": 0.7,
    "grid.alpha": 0.85,
})

def paper_axes(ax):
    ax.minorticks_on()
    ax.grid(True, which="major", linestyle=":", linewidth=0.8)
    ax.grid(True, which="minor", linestyle=":", linewidth=0.5, alpha=0.6)
    for spine in ax.spines.values():
        spine.set_linewidth(1.2)
    ax.tick_params(which="both", direction="in", top=True, right=True)

## Paths and notebook dependencies

This notebook expects the outputs from `02_session_index_and_trial_catalog.ipynb` to already exist.

In [5]:
DATASET_DIR = Path("/kaggle/input/datasets/katakuricharlotte/dandi-dataset/001201/sub-Monkey-N")        

IN_TABLE_DIR = Path("/kaggle/input/datasets/katakuricharlotte/02-session-index-results/tables_session_index")
SESSION_MASTER_CSV = IN_TABLE_DIR / "session_master_index.csv"
DECODER_TRIAL_TABLE_CSV = IN_TABLE_DIR / "decoder_trial_table.csv"

OUT_FIG_DIR = Path("/kaggle/working/figures_stream_inventory")
OUT_TABLE_DIR = Path("/kaggle/working/tables_stream_inventory")
OUT_META_DIR = Path("/kaggle/working/meta_stream_inventory")

OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
OUT_META_DIR.mkdir(parents=True, exist_ok=True)

print("DATASET_DIR:", DATASET_DIR)
print("SESSION_MASTER_CSV:", SESSION_MASTER_CSV)
print("DECODER_TRIAL_TABLE_CSV:", DECODER_TRIAL_TABLE_CSV)
print("OUT_FIG_DIR:", OUT_FIG_DIR)
print("OUT_TABLE_DIR:", OUT_TABLE_DIR)
print("OUT_META_DIR:", OUT_META_DIR)

DATASET_DIR: /kaggle/input/datasets/katakuricharlotte/dandi-dataset/001201/sub-Monkey-N
SESSION_MASTER_CSV: /kaggle/input/datasets/katakuricharlotte/02-session-index-results/tables_session_index/session_master_index.csv
DECODER_TRIAL_TABLE_CSV: /kaggle/input/datasets/katakuricharlotte/02-session-index-results/tables_session_index/decoder_trial_table.csv
OUT_FIG_DIR: /kaggle/working/figures_stream_inventory
OUT_TABLE_DIR: /kaggle/working/tables_stream_inventory
OUT_META_DIR: /kaggle/working/meta_stream_inventory


In [6]:
assert DATASET_DIR.exists(), f"Missing dataset directory: {DATASET_DIR}"
assert SESSION_MASTER_CSV.exists(), f"Missing prerequisite file: {SESSION_MASTER_CSV}"
assert DECODER_TRIAL_TABLE_CSV.exists(), f"Missing prerequisite file: {DECODER_TRIAL_TABLE_CSV}"

session_master_df = pd.read_csv(SESSION_MASTER_CSV)
decoder_trial_table = pd.read_csv(DECODER_TRIAL_TABLE_CSV)

if "session_date" in session_master_df.columns:
    session_master_df["session_date"] = pd.to_datetime(session_master_df["session_date"], errors="coerce")

print("session_master_df shape:", session_master_df.shape)
print("decoder_trial_table shape:", decoder_trial_table.shape)

display(session_master_df.head(5))
display(decoder_trial_table.head(5))

session_master_df shape: (312, 22)
decoder_trial_table shape: (117000, 18)


,file_name,file_path,session_date,identifier,session_description,session_start_time,institution,lab,subject_id,subject_species,...,n_electrodes,n_acquisition_streams,n_processing_modules,n_intervals_tables,trials_present,units_present,electrodes_present,target_style_inferred,session_index,days_since_first_session
0,sub-Monkey-N_ses-20200127_ecephys.nwb,/kaggle/input/datasets/katakuricharlotte/dandi...,2020-01-27,2020-01-27_CO_nwb,Neural and behavioral data for target style CO,2020-01-27 00:00:00+00:00,University of Michigan,Chestek Lab,Monkey N,Macaca mulatta,...,96,0,0,1,True,False,True,CO,1,0
1,sub-Monkey-N_ses-20200130_ecephys.nwb,/kaggle/input/datasets/katakuricharlotte/dandi...,2020-01-30,2020-01-30_CO_nwb,Neural and behavioral data for target style CO,2020-01-30 00:00:00+00:00,University of Michigan,Chestek Lab,Monkey N,Macaca mulatta,...,96,0,0,1,True,False,True,CO,2,3
2,sub-Monkey-N_ses-20200204_ecephys.nwb,/kaggle/input/datasets/katakuricharlotte/dandi...,2020-02-04,2020-02-04_CO_nwb,Neural and behavioral data for target style CO,2020-02-04 00:00:00+00:00,University of Michigan,Chestek Lab,Monkey N,Macaca mulatta,...,96,0,0,1,True,False,True,CO,3,8
3,sub-Monkey-N_ses-20200205_ecephys.nwb,/kaggle/input/datasets/katakuricharlotte/dandi...,2020-02-05,2020-02-05_CO_nwb,Neural and behavioral data for target style CO,2020-02-05 00:00:00+00:00,University of Michigan,Chestek Lab,Monkey N,Macaca mulatta,...,96,0,0,1,True,False,True,CO,4,9
4,sub-Monkey-N_ses-20200206_ecephys.nwb,/kaggle/input/datasets/katakuricharlotte/dandi...,2020-02-06,2020-02-06_CO_nwb,Neural and behavioral data for target style CO,2020-02-06 00:00:00+00:00,University of Michigan,Chestek Lab,Monkey N,Macaca mulatta,...,96,0,0,1,True,False,True,CO,5,10


,file_name,session_index,session_date,identifier,target_style_inferred,id,start_time,stop_time,trial_duration,trial_number,trial_count,run_id,index_target_position,mrs_target_position,target_style,trial_timeout,timeseries_n_items,file_path
0,sub-Monkey-N_ses-20200127_ecephys.nwb,1,2020-01-27,2020-01-27_CO_nwb,CO,0,8.41,9.67,1.26,6.0,64,3,0.3,0.3,CO,30000.0,6,/kaggle/input/datasets/katakuricharlotte/dandi...
1,sub-Monkey-N_ses-20200127_ecephys.nwb,1,2020-01-27,2020-01-27_CO_nwb,CO,1,9.69,11.05,1.36,7.0,69,3,0.5,0.5,CO,30000.0,6,/kaggle/input/datasets/katakuricharlotte/dandi...
2,sub-Monkey-N_ses-20200127_ecephys.nwb,1,2020-01-27,2020-01-27_CO_nwb,CO,2,11.07,12.95,1.88,8.0,95,3,0.7,0.5,CO,30000.0,6,/kaggle/input/datasets/katakuricharlotte/dandi...
3,sub-Monkey-N_ses-20200127_ecephys.nwb,1,2020-01-27,2020-01-27_CO_nwb,CO,3,12.97,13.77,0.80,9.0,41,3,0.5,0.5,CO,30000.0,6,/kaggle/input/datasets/katakuricharlotte/dandi...
4,sub-Monkey-N_ses-20200127_ecephys.nwb,1,2020-01-27,2020-01-27_CO_nwb,CO,4,13.79,17.01,3.22,10.0,162,3,0.2,0.8,CO,30000.0,6,/kaggle/input/datasets/katakuricharlotte/dandi...


## Helper functions

The following helpers safely inspect NWB objects without forcing large data reads.

In [7]:
def safe_attr(obj, attr, default=None):
    try:
        return getattr(obj, attr)
    except Exception:
        return default

def safe_len(x, default=np.nan):
    try:
        return len(x)
    except Exception:
        return default

def safe_shape(x):
    try:
        if hasattr(x, "shape") and x.shape is not None:
            return tuple(x.shape)
        return (len(x),)
    except Exception:
        return None

def to_jsonable(x):
    if isinstance(x, (str, int, float, bool)) or x is None:
        return x
    if isinstance(x, tuple):
        return list(x)
    if isinstance(x, list):
        return [to_jsonable(v) for v in x]
    if isinstance(x, dict):
        return {str(k): to_jsonable(v) for k, v in x.items()}
    return str(x)

In [8]:
def object_path(obj):
    parts = []
    seen = set()
    cur = obj
    while cur is not None and id(cur) not in seen:
        seen.add(id(cur))
        name = safe_attr(cur, "name", None)
        if name not in [None, "root"]:
            parts.append(str(name))
        cur = safe_attr(cur, "parent", None)
    return "/".join(reversed(parts))

def object_type(obj):
    ndt = safe_attr(obj, "neurodata_type", None)
    if ndt is not None:
        return str(ndt)
    return obj.__class__.__name__

def infer_container_group(path_text):
    txt = (path_text or "").lower()
    if "acquisition" in txt:
        return "acquisition"
    if "processing" in txt:
        return "processing"
    if "analysis" in txt:
        return "analysis"
    if "stimulus" in txt:
        return "stimulus"
    if "intervals" in txt:
        return "intervals"
    return "other"

In [9]:
def describe_nwb_object(obj):
    desc = {
        "name": safe_attr(obj, "name", None),
        "neurodata_type": object_type(obj),
        "path": object_path(obj),
        "unit": safe_attr(obj, "unit", None),
        "rate": safe_attr(obj, "rate", None),
        "starting_time": safe_attr(obj, "starting_time", None),
        "description": safe_attr(obj, "description", None),
        "comments": safe_attr(obj, "comments", None),
    }

    data = safe_attr(obj, "data", None)
    timestamps = safe_attr(obj, "timestamps", None)

    desc["has_data"] = data is not None
    desc["has_timestamps"] = timestamps is not None
    desc["data_shape"] = safe_shape(data)
    desc["timestamps_shape"] = safe_shape(timestamps)
    desc["container_group"] = infer_container_group(desc["path"])
    return desc

In [10]:
def candidate_score(row):
    text = " ".join([
        str(row.get("name", "")),
        str(row.get("neurodata_type", "")),
        str(row.get("path", "")),
        str(row.get("description", "")),
        str(row.get("comments", "")),
    ]).lower()

    score = 0

    positive_keywords = {
        "spike": 4,
        "waveform": 4,
        "threshold": 4,
        "electrical": 3,
        "ecephys": 3,
        "lfp": 3,
        "mua": 3,
        "multiunit": 3,
        "event": 2,
        "timeseries": 1,
    }

    negative_keywords = {
        "eye": -4,
        "gaze": -4,
        "pupil": -4,
        "fix": -3,
        "target": -3,
        "stim": -3,
        "reward": -3,
        "position": -2,
        "cursor": -2,
        "behavior": -2,
    }

    for k, v in positive_keywords.items():
        if k in text:
            score += v

    for k, v in negative_keywords.items():
        if k in text:
            score += v

    if row.get("has_data", False):
        score += 1
    if pd.notna(row.get("rate", np.nan)):
        score += 1
    if row.get("container_group") in ["acquisition", "processing"]:
        score += 1

    return score

In [11]:
def inspect_single_nwb(nwb_path):
    rows = []

    with NWBHDF5IO(str(nwb_path), mode="r", load_namespaces=True) as io:
        nwb = io.read()

        base_meta = {
            "file_name": nwb_path.name,
            "file_path": str(nwb_path),
            "identifier": safe_attr(nwb, "identifier", None),
            "session_description": safe_attr(nwb, "session_description", None),
            "session_start_time": str(safe_attr(nwb, "session_start_time", None)),
            "subject_id": safe_attr(safe_attr(nwb, "subject", None), "subject_id", None),
        }

        for obj in nwb.all_children():
            try:
                desc = describe_nwb_object(obj)
                if desc["name"] is None:
                    continue
                row = {**base_meta, **desc}
                row["candidate_score"] = candidate_score(row)
                rows.append(row)
            except Exception:
                continue

    return rows

## Build the all-object manifest

We now scan every NWB file and create one row per discovered NWB object.

In [ ]:
nwb_files = sorted(DATASET_DIR.glob("*_ecephys.nwb"))
assert len(nwb_files) > 0, "No NWB files found."

all_rows = []

for i, nwb_path in enumerate(nwb_files, start=1):
    if i % 25 == 0 or i == 1 or i == len(nwb_files):
        print(f"Inspecting {i}/{len(nwb_files)}: {nwb_path.name}")
    rows = inspect_single_nwb(nwb_path)
    all_rows.extend(rows)

all_object_manifest = pd.DataFrame(all_rows)

if "file_name" in session_master_df.columns:
    merge_cols = [c for c in [
        "file_name", "session_index", "session_date", "target_style_inferred",
        "days_since_first_session"
    ] if c in session_master_df.columns]

    all_object_manifest = all_object_manifest.merge(
        session_master_df[merge_cols].drop_duplicates("file_name"),
        on="file_name",
        how="left"
    )

all_object_manifest["candidate_flag"] = all_object_manifest["candidate_score"] >= 4
all_object_manifest["data_shape"] = all_object_manifest["data_shape"].apply(to_jsonable)
all_object_manifest["timestamps_shape"] = all_object_manifest["timestamps_shape"].apply(to_jsonable)

all_object_manifest.to_csv(OUT_TABLE_DIR / "all_object_manifest.csv", index=False)

print("all_object_manifest shape:", all_object_manifest.shape)
display(all_object_manifest.head(10))

Inspecting 1/312: sub-Monkey-N_ses-20200127_ecephys.nwb
Inspecting 25/312: sub-Monkey-N_ses-20200822_ecephys.nwb
Inspecting 50/312: sub-Monkey-N_ses-20210122_ecephys.nwb
Inspecting 75/312: sub-Monkey-N_ses-20210421_ecephys.nwb
Inspecting 100/312: sub-Monkey-N_ses-20210706_ecephys.nwb
Inspecting 125/312: sub-Monkey-N_ses-20210816_ecephys.nwb
Inspecting 150/312: sub-Monkey-N_ses-20211025_ecephys.nwb
Inspecting 175/312: sub-Monkey-N_ses-20220210_ecephys.nwb
Inspecting 200/312: sub-Monkey-N_ses-20220415_obj-y4c3g5_ecephys.nwb
Inspecting 225/312: sub-Monkey-N_ses-20220725_ecephys.nwb
Inspecting 250/312: sub-Monkey-N_ses-20220921_obj-1k78ex8_ecephys.nwb


In [ ]:
manifest_summary = (
    all_object_manifest
    .groupby(["neurodata_type", "container_group"], dropna=False)
    .size()
    .rename("n_objects")
    .reset_index()
    .sort_values("n_objects", ascending=False)
)

manifest_summary.to_csv(OUT_TABLE_DIR / "object_type_summary.csv", index=False)
display(manifest_summary.head(25))

In [ ]:
candidate_stream_manifest = (
    all_object_manifest
    .loc[all_object_manifest["candidate_flag"]]
    .sort_values(["file_name", "candidate_score"], ascending=[True, False])
    .reset_index(drop=True)
)

candidate_stream_manifest.to_csv(OUT_TABLE_DIR / "candidate_stream_manifest.csv", index=False)

print("candidate_stream_manifest shape:", candidate_stream_manifest.shape)
display(candidate_stream_manifest.head(20))

In [ ]:
top_types = manifest_summary.head(15)

fig, ax = plt.subplots(figsize=(13, 6))
ax.barh(
    top_types["neurodata_type"][::-1],
    top_types["n_objects"][::-1],
    color="#4C72B0",
    edgecolor="black",
    linewidth=0.7
)
ax.set_xlabel("Number of objects")
ax.set_ylabel("Neurodata type")
ax.set_title("Most common NWB object types across sessions", pad=10)
paper_axes(ax)

plt.tight_layout()
plt.savefig(OUT_FIG_DIR / "fig01_top_object_types.png", dpi=300, bbox_inches="tight")
plt.savefig(OUT_FIG_DIR / "fig01_top_object_types.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
candidate_counts = (
    candidate_stream_manifest
    .groupby("file_name")
    .size()
    .rename("n_candidate_streams")
    .reset_index()
)

fig, ax = plt.subplots(figsize=(13, 6))
ax.hist(
    candidate_counts["n_candidate_streams"],
    bins=30,
    color="#55A868",
    edgecolor="black",
    linewidth=0.6,
    alpha=0.9
)
ax.set_xlabel("Candidate neural streams per session")
ax.set_ylabel("Count")
ax.set_title("Distribution of candidate decoder streams across sessions", pad=10)
paper_axes(ax)

plt.tight_layout()
plt.savefig(OUT_FIG_DIR / "fig02_candidate_stream_count_distribution.png", dpi=300, bbox_inches="tight")
plt.savefig(OUT_FIG_DIR / "fig02_candidate_stream_count_distribution.pdf", dpi=300, bbox_inches="tight")
plt.show()

## Build a session-level decoder stream map

This table chooses the highest-ranked candidate stream in each session as the default input for the next notebook.

In [ ]:
priority_order = {
    "ElectricalSeries": 5,
    "SpikeEventSeries": 6,
    "TimeSeries": 2,
    "Units": 1,
}

tmp = candidate_stream_manifest.copy()
tmp["type_priority"] = tmp["neurodata_type"].map(priority_order).fillna(0)

session_decoder_stream_map = (
    tmp.sort_values(
        ["file_name", "candidate_score", "type_priority", "name"],
        ascending=[True, False, False, True]
    )
    .groupby("file_name", as_index=False)
    .first()
)

keep_cols = [c for c in [
    "file_name", "session_index", "session_date", "target_style_inferred",
    "name", "neurodata_type", "path", "unit", "rate",
    "has_data", "has_timestamps", "data_shape", "timestamps_shape",
    "candidate_score", "container_group"
] if c in session_decoder_stream_map.columns]

session_decoder_stream_map = session_decoder_stream_map[keep_cols].rename(columns={
    "name": "selected_stream_name",
    "neurodata_type": "selected_stream_type",
    "path": "selected_stream_path",
    "unit": "selected_stream_unit",
    "rate": "selected_stream_rate",
    "has_data": "selected_stream_has_data",
    "has_timestamps": "selected_stream_has_timestamps",
    "data_shape": "selected_stream_data_shape",
    "timestamps_shape": "selected_stream_timestamps_shape",
    "candidate_score": "selected_stream_score",
    "container_group": "selected_stream_group"
})

session_decoder_stream_map["stream_selected_by"] = "highest_candidate_score"

session_decoder_stream_map.to_csv(
    OUT_TABLE_DIR / "session_decoder_stream_map.csv",
    index=False
)

display(session_decoder_stream_map.head(15))

In [ ]:
stream_name_counts = (
    session_decoder_stream_map["selected_stream_name"]
    .fillna("MISSING")
    .value_counts()
    .rename_axis("selected_stream_name")
    .reset_index(name="n_sessions")
)

stream_name_counts.to_csv(OUT_TABLE_DIR / "selected_stream_name_counts.csv", index=False)
display(stream_name_counts.head(20))

In [ ]:
top_names = stream_name_counts.head(15)

fig, ax = plt.subplots(figsize=(13, 6))
ax.barh(
    top_names["selected_stream_name"][::-1],
    top_names["n_sessions"][::-1],
    color="#C44E52",
    edgecolor="black",
    linewidth=0.7
)
ax.set_xlabel("Sessions")
ax.set_ylabel("Selected stream name")
ax.set_title("Most common default decoder streams across sessions", pad=10)
paper_axes(ax)

plt.tight_layout()
plt.savefig(OUT_FIG_DIR / "fig03_selected_stream_name_counts.png", dpi=300, bbox_inches="tight")
plt.savefig(OUT_FIG_DIR / "fig03_selected_stream_name_counts.pdf", dpi=300, bbox_inches="tight")
plt.show()

## Sample session inspection

We inspect one example session to make the decoder-input selection transparent.

In [ ]:
sample_file = session_decoder_stream_map["file_name"].dropna().iloc[0]
sample_rows = (
    all_object_manifest.loc[all_object_manifest["file_name"] == sample_file]
    .sort_values(["candidate_score", "neurodata_type", "name"], ascending=[False, True, True])
    .reset_index(drop=True)
)

sample_rows.to_csv(OUT_TABLE_DIR / "sample_session_object_manifest.csv", index=False)

print("Sample file:", sample_file)
display(sample_rows.head(30))

In [ ]:
sample_candidates = sample_rows.loc[sample_rows["candidate_flag"]].copy()
display(sample_candidates.head(20))

In [ ]:
sample_plot_df = sample_rows.head(20).copy()

fig, ax = plt.subplots(figsize=(13, 7))
ax.barh(
    sample_plot_df["name"][::-1].astype(str),
    sample_plot_df["candidate_score"][::-1],
    color=np.where(sample_plot_df["candidate_flag"][::-1], "#DD8452", "#999999"),
    edge